- https://en.wikipedia.org/wiki/Historical_components_of_the_Dow_Jones_Industrial_Average

- https://jasonzweig.com/false-profits/

In [ ]:
import numpy as np
import pandas as pd
import re
import requests
import wrds

from bs4 import BeautifulSoup

In [ ]:

from nltk.util import ngrams
from functools import partial

# Get historical DJIA components

In [3]:
resp = requests.get('https://en.wikipedia.org/wiki/Historical_components_of_the_Dow_Jones_Industrial_Average')


In [37]:
tbls = re.findall(r'<h2 id=.+?>.+?</table>', resp.text, re.DOTALL)

In [41]:
dow = []
for tbl in tbls:
    soup = BeautifulSoup(tbl)
    comps = pd.DataFrame([td.text.strip() for td in soup.find_all('td')],
                         columns=['company'])
    comps['date'] = pd.to_datetime(soup.find('h2').text, errors='coerce')
    dow.append(comps)

In [42]:

dow = pd.concat(dow).reset_index(drop=True)

dow = dow.dropna()

In [43]:
dow

,company,date
0,3M Company,2024-11-08
1,"The Goldman Sachs Group, Inc.",2024-11-08
2,Nvidia Corporation ↑,2024-11-08
3,"Amazon.com, Inc.",2024-11-08
4,"The Home Depot, Inc.",2024-11-08
...,...,...
1673,The Laclede Gas Company ↑,1896-05-26
1674,The United States Leather Company (Preferred) ↑,1896-05-26
1675,Chicago Gas Light and Coke Company ↑,1896-05-26
1676,National Lead Company ↑,1896-05-26


In [45]:

# drop companies that were dropped on a given date
dow = dow[~dow['company'].str.endswith('↓')]

# clean up names
dow = dow[~(dow['company'] == 'Dropped from Average')]
dow['company'] = dow['company'].str.replace(r' ?\(.+?\)', '', regex=True)
dow['company'] = dow['company'].str.replace(r' ↑', '')
dow['company'] = dow['company'].str.replace(r' †', '')
dow['company'] = dow['company'].str.replace(r' ?\[\d\]', '', regex=True)

# drop empty company names
dow = dow[dow['company'].str.len() > 0]

In [49]:
dow = dow.sort_values(['date', 'company'])

dow = dow.set_index('date').squeeze()

In [52]:
dow.groupby(dow.index).count()

date
1896-05-26    12
1896-08-26    12
1896-11-10    12
1896-12-23    12
1898-03-24    12
1898-09-01    12
1899-04-21    12
1901-04-01    12
1901-07-01    12
1905-04-01    12
1907-11-07    12
1912-05-12    12
1915-03-16    12
1915-07-29    13
1916-10-04    20
1920-03-01    20
1924-01-22    20
1924-02-06    20
1924-05-12    20
1925-08-31    20
1925-12-07    20
1925-12-31    20
1927-03-16    20
1928-10-01    30
1929-01-08    30
1929-09-14    30
1930-01-29    30
1930-07-18    30
1932-05-26    30
1933-08-15    30
1934-08-13    30
1935-11-20    30
1939-03-04    30
1956-07-03    30
1959-06-01    30
1976-08-09    30
1979-06-29    30
1982-08-30    30
1985-10-30    30
1987-03-12    30
1991-05-06    30
1997-03-17    30
1999-11-01    30
2003-01-27    30
2004-04-08    30
2005-11-21    30
2008-02-19    30
2008-09-22    30
2009-06-08    30
2012-09-24    30
2013-09-23    30
2015-03-19    30
2017-09-01    30
2018-06-26    30
2019-04-02    30
2020-04-06    30
2020-08-31    30
2024-02-26    30
2024-11-0

In [53]:
dow = dow.loc['1925-12-31':]

In [54]:
dow.value_counts()

company
General Electric Company                       32
The Procter & Gamble Company                   31
International Business Machines Corporation    27
General Motors Corporation                     27
The Coca-Cola Company                          23
                                               ..
Victor Talking Machine Company                  1
American Can                                    1
United Drug Stores                              1
Remington Typewriter Company                    1
The Sherwin-Williams Company                    1
Name: count, Length: 128, dtype: int64

In [60]:
dow.value_counts().reset_index().drop(columns='count').to_csv('dow_companies.csv', index=True)

In [61]:
%pwd

'/Users/nstoffma/Documents/GitHub/qf/Data construction'

In [283]:
dow[dow=='American Can Company']

date
1925-12-31    American Can Company
1927-03-16    American Can Company
1929-01-08    American Can Company
1929-09-14    American Can Company
1930-01-29    American Can Company
1930-07-18    American Can Company
1932-05-26    American Can Company
1933-08-15    American Can Company
1934-08-13    American Can Company
1935-11-20    American Can Company
1939-03-04    American Can Company
1956-07-03    American Can Company
1959-06-01    American Can Company
1976-08-09    American Can Company
1979-06-29    American Can Company
1982-08-30    American Can Company
1985-10-30    American Can Company
1987-03-12    American Can Company
Name: company, dtype: object

In [12]:
# For each year, we want the date of the most recent update to the list

dates = pd.Series(dow.index.unique())

# keep last date each year
dates = dates.groupby(dates.dt.year).max()

dates = dates.reindex(index=np.arange(1925,2022))

dates = dates.fillna(method='ffill')

dates

date
1925   1925-12-31
1926   1925-12-31
1927   1927-03-16
1928   1928-10-01
1929   1929-09-14
          ...    
2017   2017-09-01
2018   2018-06-26
2019   2019-04-02
2020   2020-08-31
2021   2020-08-31
Name: date, Length: 97, dtype: datetime64[ns]

In [163]:
dow.reset_index(drop=True).drop_duplicates().to_clipboard()

# Get WRDS names data

In [346]:
conn = wrds.Connection()

Loading library list...
Done


In [265]:
msenames = conn.get_table('crsp', 'msenames', columns=['permno', 'namedt', 'nameendt', 'shrcd', 'comnam'])
msenames['permno'] = pd.to_numeric(msenames['permno'], downcast='integer')

msenames[msenames['comnam'].str.contains('INTERNATIONAL NICKEL')]

,permno,namedt,nameendt,shrcd,comnam
8479,12546,1925-12-31,1962-07-01,12.0,INTERNATIONAL NICKEL CO CDA LTD
8480,12546,1962-07-02,1968-01-01,12.0,INTERNATIONAL NICKEL CO CDA LTD
8481,12546,1968-01-02,1974-09-08,12.0,INTERNATIONAL NICKEL CO CDA LTD
8482,12546,1974-09-09,1976-04-21,12.0,INTERNATIONAL NICKEL CO CDA LTD


In [139]:
# dates for IBM

crsp_dates = conn.raw_sql("""
    SELECT DISTINCT max(date) as date
    FROM crsp.msf
    WHERE permno=12490
    GROUP BY date_trunc('year', date)
    ORDER BY date""",
    date_cols=['date']).squeeze()

Pull end-of-year price data

In [159]:
# pass dates as parameters to query
params = {'dates' : tuple(crsp_dates.dt.strftime('%m/%d/%Y').values)}

crsp = conn.raw_sql("""
    SELECT a.permno, a.date, abs(a.prc)*a.shrout as sz, b.comnam
    FROM crsp.msf a
    JOIN crsp.msenames b
    ON a.permno=b.permno AND a.date>=b.namedt AND a.date<=b.nameendt
    WHERE b.shrcd IN (10,11,12) AND a.date IN %(dates)s
    """, params=params)

crsp['permno'] = pd.to_numeric(crsp['permno'], downcast='integer')
crsp['date'] = pd.to_datetime(crsp['date'])
crsp = crsp.set_index('date')

In [160]:
crsp.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 336976 entries, 1925-12-31 to 2022-03-31
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   permno  336976 non-null  int32  
 1   sz      330201 non-null  float64
 2   comnam  336976 non-null  object 
dtypes: float64(1), int32(1), object(1)
memory usage: 9.0+ MB


In [161]:
crsp['decile'] = crsp.groupby(crsp.index)['sz'].apply(lambda x: pd.qcut(x, q=10, labels=np.arange(1,11)))

In [162]:
crsp

,permno,sz,comnam,decile
date,,,,
1925-12-31,10006,6.540000e+04,AMERICAN CAR & FDRY CO,8
1925-12-31,10022,1.120000e+04,AMERICAN SAFETY RAZOR CORP,5
1925-12-31,10030,2.340000e+04,AMERICAN BRAKE SHOE & FDRY,6
1925-12-31,10049,1.850000e+04,ABITIBI POWER & PAPER LTD,6
1925-12-31,10057,6.125000e+03,NATIONAL ACME CO,3
...,...,...,...,...
2022-03-31,93426,4.054758e+05,VISHAY PRECISION GROUP INC,5
2022-03-31,93427,3.889284e+06,FABRINET,8
2022-03-31,93429,1.219740e+07,C B O E GLOBAL MARKETS INC,9


In [163]:
crsp.loc['1925-12-31'].groupby('decile')['sz'].agg([np.min, np.max])

,amin,amax
decile,,
1,2.500,2403.000
2,2415.000,5000.000
3,5015.000,7725.000
4,7737.375,11090.500
5,11120.000,15840.000
6,15843.625,23400.000
7,23517.000,38862.000
8,39866.000,68824.875
9,69000.000,135728.000


In [164]:
crsp[crsp.decile>=8].loc['1925-12-31']

,permno,sz,comnam,decile
date,,,,
1925-12-31,10006,65400.000,AMERICAN CAR & FDRY CO,8
1925-12-31,10137,43982.750,AMERICAN WATER WORKS & ELEC INC,8
1925-12-31,10145,248292.000,ALLIED CHEMICAL & DYE CORP,10
1925-12-31,10225,130728.500,AMERICAN TOB CO,9
1925-12-31,10233,39960.000,FAMOUS PLAYERS LASKY CORP,8
...,...,...,...,...
1925-12-31,16109,85400.000,POSTUM CEREAL CO INC DEL,9
1925-12-31,17507,155331.625,CHILE COPPER CO,10
1925-12-31,18593,72395.750,TIDE WTR OIL CO,9


In [165]:
dow.loc['1925-12-31']

date
1925-12-31         Allied Chemical and Dye Corporation
1925-12-31                        American Can Company
1925-12-31            American Car and Foundry Company
1925-12-31                 American Locomotive Company
1925-12-31        American Smelting & Refining Company
1925-12-31    American Telephone and Telegraph Company
1925-12-31                    American Tobacco Company
1925-12-31                     F. W. Woolworth Company
1925-12-31                    General Electric Company
1925-12-31                  General Motors Corporation
1925-12-31             International Harvester Company
1925-12-31                           Mack Trucks, Inc.
1925-12-31          Paramount Famous Lasky Corporation
1925-12-31                Remington Typewriter Company
1925-12-31                     Sears Roebuck & Company
1925-12-31         The American Sugar Refining Company
1925-12-31                           The Texas Company
1925-12-31                United States Rubber Company
1925-

In [166]:
states = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California', 'CO': 'Colorado',
    'CT': 'Connecticut', 'DE': 'Delaware', 'DC': 'District of Columbia', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland', 'MA': 'Massachusetts',
    'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana',
    'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico',
    'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'PR': 'Puerto Rico', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia',
    'VI': 'Virgin Islands', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}

rex = r'\b(' + '|'.join(states.keys()) + ')$'

In [167]:
def st_repl(matchobj):
    if matchobj.group(0) in states.keys():
        return states[matchobj.group(0)].upper()

In [168]:
print(re.sub(rex, st_repl, 'ABC Corp IN'))
print(re.sub(rex, st_repl, 'ABC Corp NY'))
print(re.sub(rex, st_repl, 'ABC Corp IX'))

ABC Corp INDIANA
ABC Corp NEW YORK
ABC Corp IX


In [177]:
def name_std(name):
    name = re.sub(' NEW$', '', name)
    
    # combine single letters
    name = name.replace('.', '')
    name = name.replace("'", '')
    name = re.sub(r'\b(\w) & (\w)\b', r'\1&\2', name)
    name = re.sub(r'\b(\w) (?=\w\b)', r'\1', name, flags=re.IGNORECASE)
    
    name = re.sub(r' (Incorporated|Inc\.?|Limited|Ltd\.?)$', '', name, flags=re.IGNORECASE)
    name = re.sub(r'( & | )(Company|Corporation|Corp|Cor?\.?|CP)$', '', name, flags=re.IGNORECASE)
    
    # if max length, assume IN isn't Indiana
    if len(name) == 32:
        name = re.sub(r'\bIN$', '', name)

    # eg: STANDARD OIL CO NY
    name = re.sub(r'\bCO ([A-Z]{2})$', r'\1', name, flags=re.IGNORECASE)

    # state substitutions
    name = re.sub(rex, st_repl, name)
    
    name = name.replace(' & ', ' AND ')
    name = name.replace('-', ' ')
    
    name = re.sub('^The ', '', name, flags=re.IGNORECASE)
    
    return name.strip(', ')

In [170]:
dow.loc['1925-12-31'].apply(name_std)

date
1925-12-31             Allied Chemical and Dye
1925-12-31                        American Can
1925-12-31            American Car and Foundry
1925-12-31                 American Locomotive
1925-12-31      American Smelting AND Refining
1925-12-31    American Telephone and Telegraph
1925-12-31                    American Tobacco
1925-12-31                        FW Woolworth
1925-12-31                    General Electric
1925-12-31                      General Motors
1925-12-31             International Harvester
1925-12-31                         Mack Trucks
1925-12-31              Paramount Famous Lasky
1925-12-31                Remington Typewriter
1925-12-31                       Sears Roebuck
1925-12-31             American Sugar Refining
1925-12-31                               Texas
1925-12-31                United States Rubber
1925-12-31                 United States Steel
1925-12-31                       Western Union
Name: company, dtype: object

In [180]:
crsp['comnam_std'] = crsp['comnam'].apply(name_std)

## ngram match

In [181]:
def ngram_sim(s1, s2, n=2):
    s1 = s1.upper().replace(' ', '')
    s2 = s2.upper().replace(' ', '')    
    ng1 = set(ngrams(s1, n))
    ng2 = set(ngrams(s2, n))
    return len(ng1.intersection(ng2)) / len(ng1.union(ng2))

In [182]:
crsp

,permno,sz,comnam,decile,comnam_std
date,,,,,
1925-12-31,10006,6.540000e+04,AMERICAN CAR & FDRY CO,8,AMERICAN CAR AND FDRY
1925-12-31,10022,1.120000e+04,AMERICAN SAFETY RAZOR CORP,5,AMERICAN SAFETY RAZOR
1925-12-31,10030,2.340000e+04,AMERICAN BRAKE SHOE & FDRY,6,AMERICAN BRAKE SHOE AND FDRY
1925-12-31,10049,1.850000e+04,ABITIBI POWER & PAPER LTD,6,ABITIBI POWER AND PAPER
1925-12-31,10057,6.125000e+03,NATIONAL ACME CO,3,NATIONAL ACME
...,...,...,...,...,...
2022-03-31,93426,4.054758e+05,VISHAY PRECISION GROUP INC,5,VISHAY PRECISION GROUP
2022-03-31,93427,3.889284e+06,FABRINET,8,FABRINET
2022-03-31,93429,1.219740e+07,C B O E GLOBAL MARKETS INC,9,CBOE GLOBAL MARKETS


In [183]:
a = name_std('American Car and Foundry Company')
b = name_std('AMERICAN CAR & FDRY CO')

ngram_sim(a,b)

0.7777777777777778

In [184]:
dt = pd.date_range('12/31/1925', '12/31/2021', freq='A')[0]

In [185]:
crsp[crsp['decile']>=8].loc[dt, 'comnam_std']

date
1925-12-31            AMERICAN CAR AND FDRY
1925-12-31    AMERICAN WATER WORKS AND ELEC
1925-12-31          ALLIED CHEMICAL AND DYE
1925-12-31                     AMERICAN TOB
1925-12-31             FAMOUS PLAYERS LASKY
                          ...              
1925-12-31         POSTUM CEREAL CO INC DEL
1925-12-31                     CHILE COPPER
1925-12-31                     TIDE WTR OIL
1925-12-31                      WHITE MOTOR
1925-12-31            BALTIMORE AND OHIO RR
Name: comnam_std, Length: 149, dtype: object

In [186]:
dow.loc[dates[dt.year]].values

array(['Allied Chemical and Dye Corporation', 'American Can Company',
       'American Car and Foundry Company', 'American Locomotive Company',
       'American Smelting & Refining Company',
       'American Telephone and Telegraph Company',
       'American Tobacco Company', 'F. W. Woolworth Company',
       'General Electric Company', 'General Motors Corporation',
       'International Harvester Company', 'Mack Trucks, Inc.',
       'Paramount Famous Lasky Corporation',
       'Remington Typewriter Company', 'Sears Roebuck & Company',
       'The American Sugar Refining Company', 'The Texas Company',
       'United States Rubber Company', 'United States Steel Corporation',
       'Western Union Company'], dtype=object)

In [221]:
yr = 1925

In [246]:
crsp_names = crsp[crsp['decile']>=6].loc[str(yr), ['comnam_std', 'permno']]

crsp_names = dict(crsp_names.values)

In [247]:
crsp_names

{'AMERICAN CAR AND FDRY': 10006,
 'AMERICAN BRAKE SHOE AND FDRY': 10030,
 'ABITIBI POWER AND PAPER': 10049,
 'AIR REDUCTION': 10102,
 'ALL AMER CABLES': 10129,
 'AMERICAN WATER WORKS AND ELEC': 10137,
 'ALLIED CHEMICAL AND DYE': 10145,
 'ALLIS CHALMERS': 10153,
 'AMERICAN METAL': 10161,
 'AMERICAN AND FGN PWR': 10196,
 'AMERICAN BANK NOTE': 10217,
 'AMERICAN TOB': 10428,
 'FAMOUS PLAYERS LASKY': 10233,
 'AMERICAN CAN': 10241,
 'AMERICAN EXPRESS': 10284,
 'AMERICAN INTERNATIONAL': 10313,
 'NASH MOTORS': 10321,
 'AMERICAN SMLT AND REFNG': 10364,
 'AMERICAN RADIATOR': 10372,
 'AMERICAN TELEPHONE AND TELEG': 10401,
 'AMERICAN WOOLEN': 10452,
 'AMERICAN SUGAR REFNG': 10479,
 'AMERICAN STEEL FOUNDRIES': 10487,
 'ANACONDA COPPER MNG': 10495,
 'ARMOUR': 10524,
 'ASSOCIATED DRY GOODS': 10559,
 'ASSOCIATED OIL': 10567,
 'ATCHISON TOPEKA AND SANTA FE RY': 10575,
 'ATLANTIC COAST LINE RR': 10591,
 'ATLANTIC REFNG': 10604,
 'BALDWIN LOCOMOTIVE WKS': 10671,
 'BARNSDALL': 10719,
 'BEECH NUT PACKING':

In [226]:
dow_comp = dow.loc[dates[yr]].values[0]

In [227]:
dow_comp

'Allied Chemical and Dye Corporation'

In [243]:
_sim = partial(ngram_sim, name_std(dow_comp))

In [244]:
_sim('ALLIED CHEM DYE')

0.5789473684210527

In [249]:
max(crsp_names.keys(), key=_sim)

'ALLIED CHEMICAL AND DYE'

In [255]:
matches = []
for yr in range(1925,2022):
    
    crsp_names = crsp[crsp['decile']>=6].loc[str(yr), ['comnam_std', 'permno']]
    crsp_names = dict(crsp_names.values)
    
    for dow_comp in dow.loc[dates[yr]].values:
        _sim = partial(ngram_sim, name_std(dow_comp))
        match = max(crsp_names.keys(), key=_sim)
        matches.append((yr, dow_comp, match, crsp_names[match]))

In [257]:
matches = pd.DataFrame(matches, columns=['year', 'dowcomp', 'crspcomp', 'permno']).set_index('year')

In [258]:
matches.drop_duplicates().to_clipboard()

See "CRSP names" google sheet

In [136]:
crsp[crsp['permno']==19537].head()

,permno,sz,comnam,decile,comnam_std
date,,,,,
1934-12-31,19537,28693.5,NATIONAL CASH REGISTER CO,8,NATIONAL CASH REGISTER
1935-12-31,19537,37851.0,NATIONAL CASH REGISTER CO,8,NATIONAL CASH REGISTER
1936-12-31,19537,50468.0,NATIONAL CASH REGISTER CO,8,NATIONAL CASH REGISTER
1937-12-31,19537,24420.0,NATIONAL CASH REGISTER CO,8,NATIONAL CASH REGISTER
1938-12-31,19537,41107.0,NATIONAL CASH REGISTER CO,8,NATIONAL CASH REGISTER


In [266]:
msenames[msenames['permno']==10401]

,permno,namedt,nameendt,shrcd,comnam
1475,10401,1925-12-31,1962-07-01,11.0,AMERICAN TELEPHONE & TELEG CO
1476,10401,1962-07-02,1968-01-01,11.0,AMERICAN TELEPHONE & TELEG CO
1477,10401,1968-01-02,1994-04-20,11.0,AMERICAN TELEPHONE & TELEG CO
1478,10401,1994-04-21,2001-02-28,11.0,A T & T CORP
1479,10401,2001-03-01,2002-01-01,11.0,A T & T CORP
1480,10401,2002-01-02,2002-11-18,11.0,A T & T CORP
1481,10401,2002-11-19,2004-06-09,11.0,A T & T CORP
1482,10401,2004-06-10,2005-10-23,11.0,A T & T CORP
1483,10401,2005-10-24,2005-11-18,11.0,A T & T CORP


In [272]:
# AT&T reorg

conn.raw_sql("""
    SELECT event, date, nwperm
    FROM crsp.mse
    WHERE permno=10401 AND event='DELIST'
    """,
    date_cols=['date'])

,event,date,nwperm
0,DELIST,2005-11-18,66093.0


In [276]:
# GM

conn.raw_sql("""
    SELECT event, date, nwperm
    FROM crsp.mse
    WHERE permno in (12079, 66931, 68451)
    AND event='DELIST'
    """,
    date_cols=['date'])



,event,date,nwperm
0,DELIST,2009-06-01,0.0
1,DELIST,1996-06-07,83596.0
2,DELIST,2003-12-22,89954.0


In [278]:
# GM

conn.raw_sql("""
    SELECT permno, namedt, nameendt, shrcd, comnam, shrcls, cusip
    FROM crsp.msenames
    WHERE permno in (12079, 66931, 68451)
    """,
    date_cols=['date'])



,permno,namedt,nameendt,shrcd,comnam,shrcls,cusip
0,12079.0,1925-12-31,1962-07-01,11.0,GENERAL MOTORS CORP,None,37044210
1,12079.0,1962-07-02,1968-01-01,11.0,GENERAL MOTORS CORP,None,37044210
2,12079.0,1968-01-02,2002-01-01,11.0,GENERAL MOTORS CORP,None,37044210
3,12079.0,2002-01-02,2004-06-09,11.0,GENERAL MOTORS CORP,None,37044210
4,12079.0,2004-06-10,2009-06-01,11.0,GENERAL MOTORS CORP,None,37044210
5,66931.0,1984-10-19,1996-06-07,11.0,GENERAL MOTORS CORP,E,37044240
6,68451.0,1985-12-31,1997-12-17,11.0,GENERAL MOTORS CORP,H,37044283
7,68451.0,1997-12-18,2002-01-01,11.0,GENERAL MOTORS CORP,H,37044283
8,68451.0,2002-01-02,2003-12-22,11.0,GENERAL MOTORS CORP,H,37044283


In [279]:
conn.raw_sql("""
    SELECT permno, namedt, nameendt, shrcd, comnam, shrcls, cusip
    FROM crsp.msenames
    WHERE permno in (26403, 87436)
    """,
    date_cols=['date'])


,permno,namedt,nameendt,shrcd,comnam,shrcls,cusip
0,26403.0,1957-11-12,1962-07-01,11.0,DISNEY WALT PRODUCTIONS,None,25468710
1,26403.0,1962-07-02,1968-01-01,11.0,DISNEY WALT PRODUCTIONS,None,25468710
2,26403.0,1968-01-02,1986-02-05,11.0,DISNEY WALT PRODUCTIONS,None,25468710
3,26403.0,1986-02-06,1996-02-11,11.0,DISNEY WALT CO,None,25468710
4,26403.0,1996-02-12,2001-09-27,11.0,DISNEY WALT CO,None,25468710
5,26403.0,2001-09-28,2002-01-01,11.0,DISNEY WALT CO,None,25468710
6,26403.0,2002-01-02,2004-06-09,11.0,DISNEY WALT CO,None,25468710
7,26403.0,2004-06-10,2014-01-26,11.0,DISNEY WALT CO,None,25468710
8,26403.0,2014-01-27,2016-12-18,11.0,DISNEY WALT CO,None,25468710
9,26403.0,2016-12-19,2021-02-28,11.0,DISNEY WALT CO,None,25468710


In [288]:
conn.raw_sql("""
    SELECT permno, date, ret, retx, prc, shrout
    FROM crsp.msf
    WHERE permno in (26403, 87436) AND date='2000-01-31'
    """,
    date_cols=['date'])


,permno,date,ret,retx,prc,shrout
0,26403.0,2000-01-31,0.241453,0.241453,36.3125,2095320.0
1,87436.0,2000-01-31,0.078947,0.078947,25.6250,43364.0


### Problems

- National Cash Register starts in 1934

In [290]:
msenames[msenames.comnam.str.contains('CASH REG')]

,permno,namedt,nameendt,shrcd,comnam
23634,19537,1934-04-26,1962-07-01,11.0,NATIONAL CASH REGISTER CO
23635,19537,1962-07-02,1966-01-02,11.0,NATIONAL CASH REGISTER CO
23636,19537,1966-01-03,1968-01-01,11.0,NATIONAL CASH REGISTER CO
23637,19537,1968-01-02,1974-05-12,11.0,NATIONAL CASH REGISTER CO


In [291]:
%pwd

'/Users/nstoffma/Dropbox/Teaching/Online course/Notebooks'

**NOTE:** I dropped one of the share classes for American Tobacco. Could do something else. Also General Motors.

In [396]:
permnos = pd.read_csv('/Users/nstoffma/Dropbox/Teaching/QF Book/Dow comps.tsv',
                      sep='\t', header=None, names=['company', 'permno']
                     ).drop_duplicates()

In [397]:
permnos

,company,permno
0,3M Company,22592
1,Alcoa Inc.,24643
2,Allied Chemical and Dye Corporation,10145
3,Allied Chemical Corporation,10145
4,Allied-Signal Incorporated,10145
...,...,...
119,"Walgreens Boots Alliance, Inc.",19502
120,Walmart Inc.,55976
121,Western Union Company,15325
122,Westinghouse Electric Corporation,15368


In [401]:
m = pd.merge(dow.reset_index(), permnos, how='left')         

In [402]:
m[m.permno.isna()]

,date,company,permno
86,1929-01-08,National Cash Register Company,NaN
118,1929-09-14,National Cash Register Company,NaN
149,1930-01-29,National Cash Register Company,NaN
178,1930-07-18,National Cash Register Company,NaN
967,2018-02-01,"Verizon Communications, Inc.",NaN
1047,2020-04-06,Raytheon Technologies Corporation,NaN


In [403]:
# Verizon
m.loc[967, 'permno'] = 65875

# Raytheon
m.loc[1047, 'permno'] = 17830

In [404]:
m[m.permno.isna()]

,date,company,permno
86,1929-01-08,National Cash Register Company,NaN
118,1929-09-14,National Cash Register Company,NaN
149,1930-01-29,National Cash Register Company,NaN
178,1930-07-18,National Cash Register Company,NaN


In [405]:
m = m.dropna()

m['permno'] = pd.to_numeric(m['permno'], downcast='integer')

In [406]:
m

,date,company,permno
0,1925-12-31,Allied Chemical and Dye Corporation,10145
1,1925-12-31,American Can Company,10241
2,1925-12-31,American Car and Foundry Company,10006
3,1925-12-31,American Locomotive Company,11287
4,1925-12-31,American Smelting & Refining Company,10364
...,...,...,...
1085,2020-08-31,UnitedHealth Group Incorporated,92655
1086,2020-08-31,Verizon Communications Inc.,65875
1087,2020-08-31,Visa Inc.,92611
1088,2020-08-31,"Walgreens Boots Alliance, Inc.",19502


In [407]:
m.groupby('date')['permno'].count()

date
1925-12-31    20
1927-03-16    20
1928-10-01    30
1929-01-08    29
1929-09-14    29
1930-01-29    29
1930-07-18    29
1932-05-26    30
1933-08-15    30
1934-08-13    30
1935-11-20    30
1939-03-04    30
1956-07-03    30
1959-06-01    30
1976-08-09    30
1979-06-29    30
1982-08-30    30
1985-10-30    30
1987-03-12    30
1991-05-06    30
1997-03-17    30
1999-11-01    30
2003-01-27    30
2004-04-08    30
2005-11-21    30
2008-02-19    30
2008-09-22    30
2009-06-08    30
2012-09-24    30
2013-09-23    30
2015-03-19    30
2017-09-01    30
2018-02-01    30
2018-06-26    30
2019-04-02    30
2020-04-06    30
2020-08-31    30
Name: permno, dtype: int64

In [413]:
%cd ~/Dropbox/Teaching/QF Book

/Users/nstoffma/Dropbox/Teaching/QF Book


In [415]:
m.to_csv('Dow components.csv', index=False)

# Returns and dividends

In [1]:
%cd ~/Dropbox/Teaching/QF Book

/Users/nstoffma/Dropbox/Teaching/QF Book


In [284]:
conn = wrds.Connection()

Loading library list...
Done


In [4]:
dow = pd.read_csv('Dow components.csv', parse_dates=['date'], index_col='date')

In [5]:
dates = pd.Series(dow.index.unique())

# keep last date each year
dates = dates.groupby(dates.dt.year).max()

dates = dates.reindex(index=np.arange(1925,2022))

dates = dates.fillna(method='ffill')

dates.index.name = 'year'

dates

year
1925   1925-12-31
1926   1925-12-31
1927   1927-03-16
1928   1928-10-01
1929   1929-09-14
          ...    
2017   2017-09-01
2018   2018-06-26
2019   2019-04-02
2020   2020-08-31
2021   2020-08-31
Name: date, Length: 97, dtype: datetime64[ns]

In [6]:
dow_comps = pd.merge(dates.reset_index(), dow.reset_index()).drop('date', axis=1)

In [7]:
dow_comps

,year,company,permno
0,1925,Allied Chemical and Dye Corporation,10145
1,1925,American Can Company,10241
2,1925,American Car and Foundry Company,10006
3,1925,American Locomotive Company,11287
4,1925,American Smelting & Refining Company,10364
...,...,...,...
2872,2021,UnitedHealth Group Incorporated,92655
2873,2021,Verizon Communications Inc.,65875
2874,2021,Visa Inc.,92611
2875,2021,"Walgreens Boots Alliance, Inc.",19502


In [292]:
dow_comps[dow_comps.company=='American Can Company']

,year,company,permno
1,1925,American Can Company,10241
21,1926,American Can Company,10241
41,1927,American Can Company,10241
91,1929,American Can Company,10241
120,1930,American Can Company,10241
...,...,...,...
1799,1986,American Can Company,10241
1829,1987,American Can Company,10241
1859,1988,American Can Company,10241
1889,1989,American Can Company,10241


In [299]:
conn.raw_sql("select * from crsp.mse where permno=10241 and event='DELIST'").T.fillna(np.nan).dropna()

,0
event,DELIST
date,1988-12-15
hsicmg,34.0
hsicig,341.0
cusip,74158710
dlamt,28.75
dlpdt,1988-12-16
dlstcd,241.0
hsiccd,3411.0
issuno,0.0


In [300]:
conn.raw_sql("select * from crsp.msenames where permno=70519")

,permno,namedt,nameendt,shrcd,exchcd,siccd,ncusip,ticker,comnam,shrcls,...,naics,primexch,trdstat,secstat,permco,compno,issuno,hexcd,hsiccd,cusip
0,70519.0,1986-10-29,1988-05-05,11.0,1.0,6153.0,20161510,CCC,COMMERCIAL CREDIT CO,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
1,70519.0,1988-05-06,1988-12-15,11.0,1.0,6153.0,20161510,CCC,COMMERCIAL CREDIT GROUP INC,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
2,70519.0,1988-12-16,1989-01-19,11.0,1.0,6021.0,74158910,CCC,PRIMERICA CORP NEW,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
3,70519.0,1989-01-20,1994-01-02,11.0,1.0,5999.0,74158910,PA,PRIMERICA CORP NEW,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
4,70519.0,1994-01-03,1995-04-26,11.0,1.0,6211.0,89419010,TRV,TRAVELERS INC,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
5,70519.0,1995-04-27,1998-10-07,11.0,1.0,6211.0,89419010,TRV,TRAVELERS GROUP INC,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
6,70519.0,1998-10-08,1998-12-03,11.0,1.0,6331.0,17296710,CCI,CITIGROUP INC,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
7,70519.0,1998-12-04,2000-05-31,11.0,1.0,6153.0,17296710,C,CITIGROUP INC,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
8,70519.0,2000-06-01,2002-01-01,11.0,1.0,6211.0,17296710,C,CITIGROUP INC,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742
9,70519.0,2002-01-02,2002-01-31,11.0,1.0,6211.0,17296710,C,CITIGROUP INC,None,...,None,N,A,R,20483.0,99995023.0,0.0,1.0,6021.0,17296742


**Note:** NCR is missing so only have 29 firms until 1932.

In [244]:
dow_comps.groupby('year')['year'].count().loc[lambda x: x!=30]

year
1925    20
1926    20
1927    20
1929    29
1930    29
1931    29
Name: year, dtype: int64

## Returns

In [206]:
# restrict queries to permnos that are in the Dow at some point

params = {'permnos': (tuple([str(permno) for permno in np.unique(dow['permno'])]))}

In [207]:
rets = conn.raw_sql("""
    SELECT permno, mthcaldt, mthprc, mthprevprc, mthret, mthretx
    FROM crsp.msf_v2
    WHERE permno IN %(permnos)s
    """,
    date_cols=['mthcaldt'], params=params)

rets['permno'] = pd.to_numeric(rets['permno'], downcast='integer')

rets = rets.sort_values(['permno', 'mthcaldt'])

In [209]:
annrets = rets.groupby(['permno', rets['mthcaldt'].dt.year])[['mthret', 'mthretx']].apply(lambda x: (1+x).product()-1)

prcs = rets.groupby(['permno', rets['mthcaldt'].dt.year])[['mthprc']].last()

annrets = annrets.join(prcs).reset_index()

annrets = annrets.rename(columns={'mthcaldt':'year', 'mthret':'ret', 'mthretx':'retx', 'mthprc':'prc'})

In [210]:
annrets

,permno,year,ret,retx,prc
0,10006,1925,0.000000,0.000000,109.000
1,10006,1926,0.008753,-0.049231,101.500
2,10006,1927,0.150657,0.086207,110.250
3,10006,1928,-0.054262,-0.109977,98.125
4,10006,1929,-0.153677,-0.205096,78.000
...,...,...,...,...,...
6012,92655,2018,0.145207,0.130001,249.120
6013,92655,2019,0.199871,0.180074,293.980
6014,92655,2020,0.212048,0.192870,350.680
6015,92655,2021,0.452131,0.431904,502.140


## Dividends

This appears to be more accurate than using `ret - retx`.

In [211]:
sql = """
    SELECT permno, date_part('year', exdt) as year, distcd, sum(divamt) as div
    FROM crsp.msedist
    WHERE permno IN %(permnos)s
        AND divamt>0
    GROUP BY permno, year, distcd
    ORDER BY permno, year, distcd
    """

divs = conn.raw_sql(sql, params=params)
divs = divs.dropna()

divs[['permno', 'year', 'distcd']] = divs[['permno', 'year', 'distcd']].apply(lambda x: pd.to_numeric(x, downcast='integer'))

divs['distcd'] = divs['distcd'].astype(str)

In [212]:
# aggregate all dividend types

divs = divs.groupby(['permno', 'year'])['div'].sum().reset_index()

In [213]:
divs.head()

,permno,year,div
0,10006,1926,8.3125
1,10006,1927,6.0000
2,10006,1928,6.0000
3,10006,1929,6.0000
4,10006,1930,6.0000


In [214]:
annrets.head()

,permno,year,ret,retx,prc
0,10006,1925,0.000000,0.000000,109.000
1,10006,1926,0.008753,-0.049231,101.500
2,10006,1927,0.150657,0.086207,110.250
3,10006,1928,-0.054262,-0.109977,98.125
4,10006,1929,-0.153677,-0.205096,78.000


In [253]:
annretsdivs = pd.merge(annrets, divs, how='left')
annretsdivs['div'] = annretsdivs['div'].fillna(0)

In [254]:
annretsdivs

,permno,year,ret,retx,prc,div
0,10006,1925,0.000000,0.000000,109.000,0.0000
1,10006,1926,0.008753,-0.049231,101.500,8.3125
2,10006,1927,0.150657,0.086207,110.250,6.0000
3,10006,1928,-0.054262,-0.109977,98.125,6.0000
4,10006,1929,-0.153677,-0.205096,78.000,6.0000
...,...,...,...,...,...,...
6012,92655,2018,0.145207,0.130001,249.120,3.4500
6013,92655,2019,0.199871,0.180074,293.980,4.1400
6014,92655,2020,0.212048,0.192870,350.680,4.8300
6015,92655,2021,0.452131,0.431904,502.140,5.6000


In [264]:
df = dow_comps.merge(annretsdivs, how='left')

In [265]:
df = df[df['year']>1925]

In [266]:
df

,year,company,permno,ret,retx,prc,div
20,1926,Allied Chemical and Dye Corporation,10145,0.234457,0.194079,136.125,4.0000
21,1926,American Can Company,10241,0.049807,0.005128,49.000,5.7500
22,1926,American Car and Foundry Company,10006,0.008753,-0.049231,101.500,8.3125
23,1926,American Locomotive Company,11287,-0.000484,-0.073016,109.500,8.0000
24,1926,American Smelting & Refining Company,10364,0.045476,-0.012111,142.750,7.2500
...,...,...,...,...,...,...,...
2872,2021,UnitedHealth Group Incorporated,92655,0.452131,0.431904,502.140,5.6000
2873,2021,Verizon Communications Inc.,65875,-0.075320,-0.115574,51.960,2.5225
2874,2021,Visa Inc.,92611,-0.003208,-0.009235,216.710,1.3350
2875,2021,"Walgreens Boots Alliance, Inc.",19502,0.358067,0.307924,52.160,1.8900


In [267]:
df.groupby('year')['year'].count().loc[lambda x: x!=30]

year
1926    20
1927    20
1929    29
1930    29
1931    29
Name: year, dtype: int64

In [270]:
df[df['ret'].isna()]

,year,company,permno,ret,retx,prc,div
1889,1989,American Can Company,10241,NaN,NaN,NaN,NaN
1919,1990,American Can Company,10241,NaN,NaN,NaN,NaN
2262,2001,J.P. Morgan & Company,48071,NaN,NaN,NaN,NaN
2292,2002,J.P. Morgan & Company,48071,NaN,NaN,NaN,NaN


In [274]:
conn.raw_sql("""
    select * from crsp.msenames where permno=10241
    """)

,permno,namedt,nameendt,shrcd,exchcd,siccd,ncusip,ticker,comnam,shrcls,...,naics,primexch,trdstat,secstat,permco,compno,issuno,hexcd,hsiccd,cusip
0,10241.0,1925-12-31,1962-07-01,10.0,1.0,3410.0,None,None,AMERICAN CAN CO,None,...,None,N,A,R,22177.0,0.0,0.0,1.0,3411.0,74158710
1,10241.0,1962-07-02,1968-01-01,10.0,1.0,3411.0,None,AC,AMERICAN CAN CO,None,...,None,N,A,R,22177.0,0.0,0.0,1.0,3411.0,74158710
2,10241.0,1968-01-02,1987-04-27,10.0,1.0,3411.0,02484310,AC,AMERICAN CAN CO,None,...,None,N,A,R,22177.0,0.0,0.0,1.0,3411.0,74158710
3,10241.0,1987-04-28,1988-12-15,10.0,1.0,3411.0,74158710,PA,PRIMERICA CORP,None,...,None,N,A,R,22177.0,0.0,0.0,1.0,3411.0,74158710


In [276]:
dow_comps[dow_comps.permno==10241]

,year,company,permno
1,1925,American Can Company,10241
21,1926,American Can Company,10241
41,1927,American Can Company,10241
61,1928,American Can,10241
91,1929,American Can Company,10241
...,...,...,...
1799,1986,American Can Company,10241
1829,1987,American Can Company,10241
1859,1988,American Can Company,10241
1889,1989,American Can Company,10241


In [17]:
dowcomps = []

for eoy in pd.date_range('1925-12-31', '2020-12-31', freq='A'):
    yr = int(eoy.strftime('%Y'))
    dt = dates[yr].strftime('%Y-%m-%d')
    params = {'permnos':  (tuple([str(permno) for permno in dow.loc[dt, 'permno']]))}
    sql = """
        SELECT permno, mthcaldt, mthprc, mthprevprc, mthret, mthretx
        FROM crsp.msf_v2
        WHERE date_part('year', mthcaldt)={} AND permno IN %(permnos)s
        """.format(yr)
    
    df = conn.raw_sql(sql, params=params, date_cols=['date'])
    dowcomps.append(df)

dowcomps = pd.concat(dowcomps)

dowcomps['permno'] = pd.to_numeric(dowcomps['permno'], downcast='integer')

dowcomps = dowcomps.sort_values(['permno', 'date'])

# Experimenting

# Clean

In [15]:
from collections import Counter

In [ ]:
charcnts = Counter()

for comp in crsp['comnam']:
    for char in list(comp):
        charcnts[char] += 1

In [ ]:
charcnts.most_common()

In [ ]:
crsp[crsp['comnam'].str.contains('-')]

In [ ]:
crsp[crsp['comnam'].str.contains('AMAZON')]

In [ ]:
crsp[crsp['permno']==55976]

In [ ]:
def name_std(name):

    name = name.replace('.', '')

    # combine single letters
    name = re.sub(r'\b(\w) & (\w)\b', r'\1&\2', name)
    name = re.sub(r'\b(\w) (?=\w\b)', r'\1', name, flags=re.IGNORECASE)
    
    return name.strip()

In [ ]:
crsp['comnam_std'] = crsp['comnam'].apply(name_std)

In [ ]:
crsp['comnam_std'].iloc[0].split()[-1]

In [ ]:
wordcnts = Counter()

for comp in crsp['comnam_std']:
    wordcnts[comp.split()[-1]] += 1

In [ ]:
wordcnts = pd.Series(wordcnts).sort_values(ascending=False)

In [ ]:
wordcnts.iloc[:25]

In [ ]:
wordcnts.to_clipboard()

In [ ]:
crsp[crsp['comnam_std'].str.contains(r'\d$')]

In [ ]:
crsp[crsp['comnam_std'].str.contains(r'\bCP$')]

In [ ]:
Counter(crsp['comnam_std'].str.extract(r'\b(\w{2})$').dropna().squeeze().values).most_common()

In [ ]:
crsp[crsp['comnam_std'].str.contains(r'\bIN$')]

In [ ]:
crsp['cnlen'] = crsp['comnam_std'].str.len()

In [ ]:
crsp[crsp['comnam_std'].str.contains(r'\bIN$')]

In [ ]:
crsp['cnlen'].describe()

In [ ]:
crsp[crsp['cnlen']==2]

In [ ]:
crsp[crsp.permno==13687]

In [116]:
def name_std(name):
    name = re.sub(' NEW$', '', name)
    
    # combine single letters
    name = name.replace('.', '')    
    name = re.sub(r'\b(\w) & (\w)\b', r'\1&\2', name)
    name = re.sub(r'\b(\w) (?=\w\b)', r'\1', name, flags=re.IGNORECASE)
    
    name = re.sub(r' (Incorporated|Inc\.?|Limited|Ltd\.?)$', '', name, flags=re.IGNORECASE)
    name = re.sub(r'( & | )(Company|Corporation|Corp|Cor?\.?|CP)$', '', name, flags=re.IGNORECASE)
    
    # if max length, assume IN isn't Indiana
    if len(name) == 32:
        name = re.sub(r'\bIN$', '', name)

    # eg: STANDARD OIL CO NY
    name = re.sub(r'\bCO ([A-Z]{2})$', r'\1', name, flags=re.IGNORECASE)

    # state substitutions
    name = re.sub(rex, st_repl, name)
    
    name = name.replace(' & ', ' AND ')
    name = name.replace('-', ' ')
    
    return name.strip()

In [117]:
tests = [
    'AMERICAN TELEPHONE & TELEG',
    'American Telephone and Telegraph Company',
    'International Business Machines Corporation',
    'Standard Oil Co. of New Jersey',
    'Standard Oil Co. of California',
    'MORGAN J P & CO INC',
    'J.P. Morgan & Co.',
    'STANDARD OIL CO N Y',
    'ABC & Co',
    'ABCCorp',
    'ABC Corp'
]

In [118]:
for test in tests:
    print('{}\t-->\t{}'.format(
        test,
        name_std(test)
    ))

AMERICAN TELEPHONE & TELEG	-->	AMERICAN TELEPHONE AND TELEG
American Telephone and Telegraph Company	-->	American Telephone and Telegraph
International Business Machines Corporation	-->	International Business Machines
Standard Oil Co. of New Jersey	-->	Standard Oil Co of New Jersey
Standard Oil Co. of California	-->	Standard Oil Co of California
MORGAN J P & CO INC	-->	MORGAN JP
J.P. Morgan & Co.	-->	JP Morgan
STANDARD OIL CO N Y	-->	STANDARD OIL NEW YORK
ABC & Co	-->	ABC
ABCCorp	-->	ABCCorp
ABC Corp	-->	ABC


In [119]:
crsp['comnam_std'] = crsp['comnam'].apply(name_std)

In [120]:
crsp

,permno,namedt,nameendt,ticker,comnam,comnam_std
0,77613,1992-05-15,1998-05-27,OISI,OPHTHALMIC IMAGING SYSTEMS,OPHTHALMIC IMAGING SYSTEMS
1,80607,1994-06-24,1995-04-05,ACBI,ATLANTIC COMMUNITY BANCORP INC,ATLANTIC COMMUNITY BANCORP
2,59942,1972-12-14,1977-07-18,MSRX,MEASUREX CORP,MEASUREX
3,21389,1940-10-26,1962-07-01,NaN,PITTSBURGH FORGINGS CO,PITTSBURGH FORGINGS
4,61138,2004-06-01,2012-09-28,PRX,PAR PHARMACEUTICAL COMPANIES INC,PAR PHARMACEUTICAL COMPANIES
...,...,...,...,...,...,...
41805,80841,1994-08-12,1999-06-08,SIRN,SIRENA APPAREL GROUP INC,SIRENA APPAREL GROUP
41806,87402,1999-11-30,2001-08-10,NBCI,N B C INTERNET INC,NBC INTERNET
41807,76300,1993-06-03,1995-06-30,ASFL,AMERICAN SAVINGS OF FLORIDA FSB,AMERICAN SAVINGS OF FLORIDA FSB
41808,50279,1988-08-31,1992-05-17,NaN,LUNN INDUSTRIES INC,LUNN INDUSTRIES


In [121]:
# Count unique permnos per word

from collections import defaultdict

words = defaultdict(list)

for i, rec in crsp.iterrows():
    for w in rec['comnam_std'].split():
        words[w].append(rec['permno'])

In [122]:
scores = Counter()
for word in words:
    scores[word] = 1 / len(set(words[word]))
        

In [126]:
for w in ['INTERNATIONAL', 'AMAZON', 'BUSINESS']:
    print(scores[w])

0.001176470588235294
1.0
0.016129032258064516


In [80]:
def first3(name):
    return([w[:3] for w in name.split()])

In [81]:
first3('International Business Machs')

['Int', 'Bus', 'Mac']

In [130]:
crsp['f3'] = crsp['comnam_std'].apply(lambda x: '|'.join(first3(x)))

In [132]:
crsp['f3'].value_counts()

INT                132
COM                 86
MED                 76
CON                 72
BIO                 68
                  ... 
COM|DIM              1
HER|BAN|ANA|CAL      1
WGN                  1
AUL|GLO|HOL          1
NBC|INT              1
Name: f3, Length: 27023, dtype: int64

In [136]:
dow.head()

date
1925-12-31     Allied Chemical and Dye Corporation
1925-12-31                    American Can Company
1925-12-31        American Car and Foundry Company
1925-12-31             American Locomotive Company
1925-12-31    American Smelting & Refining Company
Name: company, dtype: object

In [137]:
crsp[crsp['comnam'].str.startswith('ALLIED CHE')]

,permno,namedt,nameendt,ticker,comnam,comnam_std,f3
30873,10145,1962-07-02,1981-04-27,ACD,ALLIED CHEMICAL CORP,ALLIED CHEMICAL,ALL|CHE
31181,10145,1925-12-31,1958-04-27,NaN,ALLIED CHEMICAL & DYE CORP,ALLIED CHEMICAL AND DYE,ALL|CHE|AND|DYE
36589,10145,1958-04-28,1962-07-01,NaN,ALLIED CHEMICAL CORP,ALLIED CHEMICAL,ALL|CHE


In [142]:
name_std('F. W. Woolworth Company')

'FW Woolworth'

In [143]:
name_std('WOOLWORTH F W')

'WOOLWORTH FW'

In [147]:
wordlist('American Telephone and Telegraph Company')

{'American', 'Company', 'Telegraph', 'Telephone', 'and'}

In [152]:
first3('American Telephone and Telegraph Company')

['Ame', 'Tel', 'and', 'Tel', 'Com']

In [154]:
def sim(comp, cand):
    comp = name_std(comp.upper())
    cand = name_std(cand.upper())
        
    # Exact match
    if comp == cand:
        return 1
    
    # Same words in a different order
    if set(comp.split()) == set(comp.split()):
        return 0.95
        
    # Same first three letters by word
    if first3(comp) == first3(cand):
        return 0.9
    
    return 0

In [168]:
sim('American Telephone and Telegraph Company',
    'AMERICAN TELEPHONE & TELEG CO')

0.95

In [84]:
def is_abbrev(a, b):
    '''Returns True if b is a possible abbreviation of a'''
    
    if len(b) == 0: return True
 
    if len(a) == 0: return False
 
    if(a[0] == b[0]):
        return is_abbrev(a[1:], b[1:])
    else:
        return is_abbrev(a[1:], b)

In [85]:
for p in [('international', 'intl'), ('international', 'itnl'), ('machines', 'machs'), ('machines', 'mahc')]:
    print(p, is_abbrev(*p))

('international', 'intl') True
('international', 'itnl') True
('machines', 'machs') True
('machines', 'mahc') False


for each word in a, check if a word in b is exact match

In [155]:
dow.head()

date
1925-12-31     Allied Chemical and Dye Corporation
1925-12-31                    American Can Company
1925-12-31        American Car and Foundry Company
1925-12-31             American Locomotive Company
1925-12-31    American Smelting & Refining Company
Name: company, dtype: object

In [170]:
dts = pd.date_range('12/31/1925', '12/31/2021', freq='A')

In [176]:
dt = dts[0]

In [177]:
crsp_names = crsp.loc[(crsp.namedt<=dt) & (crsp.nameendt>=dt), 'comnam_std']

In [178]:
crsp_names

54                   SHELL UNION OIL
62                BRITISH EMPIRE STL
173        WHEELING AND LAKE ERIE RY
223         MISSOURI KANSAS TEXAS RR
239                   AMERICAN METAL
                    ...             
40533        KANSAS CITY SOUTHERN RY
40617        STANDARD OIL NEW JERSEY
41050    UNIVERSAL PIPE AND RADIATOR
41420        REPUBLIC IRON AND STEEL
41618            LEE RUBBER AND TIRE
Name: comnam_std, Length: 497, dtype: object

In [181]:
dow_comp = 'American Smelting & Refining Company'

name_std(dow_comp)

'American Smelting AND Refining'

In [ ]:
matches = []
for dt in pd.date_range('12/31/1925', '12/31/2021', freq='A'):
    # select companies that existed on “dt”
    crsp_names = crsp.loc[(crsp.namedt<=dt) & (crsp.nameendt>=dt), 'comnam_std']
    for dow_comp in dow.loc[dates[dt.year]].values:
        _sim = partial(ngram_sim, name_std(dow_comp))
        matches.append((dt, dow_comp, max(crsp_names, key=_sim)))

In [92]:
def sim(a, b):
    a = name_std(a.upper())
    b = name_std(b.upper())
    
    same = wordlist(a).intersection(wordlist(b))
    
    print(same)
    
    score = sum([scores[word] for word in same])
    return score

In [93]:
sim('American Telephone and Telegraph Company',
    'AMERICAN TELEPHONE & TELEG CO')

{'AMERICAN', 'TELEPHONE'}


0.040021600864034564

In [ ]:
def sim(a, b):
    a = name_std(a.upper())
    b = name_std(b.upper())
    
    words_a = a.split()
    words_b = b.split()
    
    matches = []
    for wrd in words_a:
        for i in range(len(words_b)):
            if wrd == words_b[i]:
                matches.append((wrd, words_b[i]))
    
    print(matches)

In [ ]:
sim('American Telephone and Telegraph Company',
    'AMERICAN TELEPHONE & TELEG CO')

In [ ]:
crsp['comnam_std'] = crsp['comnam'].apply(name_std)

In [ ]:
crsp[crsp.ticker.isin(['AAPL', 'IBM', 'T'])]

In [ ]:
crsp[crsp['comnam'].str.contains(r'\bCO \w ?\w$')]

In [ ]:
crsp[crsp['comnam'].str.contains('STANDARD OIL')]

# Fuzzy merge

- put single letters together
- word-level
- ngram
- acronyms

In [ ]:
def ngram_sim(s1, s2, n=2):
    s1 = s1.upper().replace(' ', '')
    s2 = s2.upper().replace(' ', '')    
    ng1 = set(ngrams(s1, n))
    ng2 = set(ngrams(s2, n))
    return len(ng1.intersection(ng2)) / len(ng1.union(ng2))

In [ ]:
matches = []
for dt in pd.date_range('12/31/1925', '12/31/2021', freq='A'):
    # select companies that existed on “dt”
    crsp_names = crsp.loc[(crsp.namedt<=dt) & (crsp.nameendt>=dt), 'comnam_std']
    for dow_comp in dow.loc[dates[dt.year]].values:
        _sim = partial(ngram_sim, name_std(dow_comp))
        matches.append((dt, dow_comp, max(crsp_names, key=_sim)))

In [ ]:
df = pd.DataFrame(matches, columns=['date', 'name', 'crsp_name']).set_index('date')

df

In [ ]:
df.loc['1929']

In [ ]:
df[['name', 'crsp_name']].drop_duplicates().to_clipboard()

In [ ]:
crsp[crsp['comnam'].str.contains('STANDARD OIL')]

In [ ]:
crsp[crsp['comnam'].str.contains('N C R')]

In [ ]:
crsp[crsp.permno==19537]

In [ ]:
dow[dow.str.contains('Standard')]

In [ ]:
pd.Series(list(set(dow.values))).to_clipboard()